# Lab Assignment — Support Vector Machine (SVM)

In [1]:
# Import pustaka yang dibutuhkan
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Lokasi data
def find_data_dir():
    # cari folder yang memuat voice.csv (cwd atau folder induknya)
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "voice.csv").exists():
            return base
    return Path.cwd()


BASE_DIR = find_data_dir()
csv_path = BASE_DIR / "voice.csv"
image_root = BASE_DIR / "images" / "images"

print("Base folder  :", BASE_DIR)
print("CSV ditemukan:", csv_path.exists())
print("Folder gambar ditemukan:", image_root.exists())

Base folder  : C:\Users\hp\JS06
CSV ditemukan: True
Folder gambar ditemukan: True


## Bagian 1 — SVM pada `voice.csv`

### 1.1 Memuat dan mengecek data

In [2]:
voice_df = pd.read_csv(csv_path)

print("Jumlah data :", voice_df.shape)
print("Jumlah fitur:", voice_df.shape[1] - 1)
print("\nDistribusi kelas:")
print(voice_df["label"].value_counts())

voice_df.head()

Jumlah data : (3168, 21)
Jumlah fitur: 20

Distribusi kelas:
label
male      1584
female    1584
Name: count, dtype: int64


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [3]:
# Pisahkan fitur dan target
X = voice_df.drop(columns=["label"])
y = voice_df["label"]

# Train SVM pada 2 rasio split x 3 kernel, lalu simpan accuracy-nya
voice_results = []

for test_size, split_name in [(0.30, "70:30"), (0.20, "80:20")]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    for kernel in ["linear", "poly", "rbf"]:
        # Scaling wajib: kernel RBF & poly butuh fitur berskala seragam
        model = make_pipeline(StandardScaler(), SVC(kernel=kernel))
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        voice_results.append(
            {
                "Split": split_name,
                "Kernel": kernel,
                "Train": len(X_train),
                "Test": len(X_test),
                "Accuracy": accuracy_score(y_test, y_pred),
            }
        )

voice_results_df = pd.DataFrame(voice_results)
voice_results_df["Accuracy (%)"] = (voice_results_df["Accuracy"] * 100).round(2)

print("=== Ringkasan Accuracy SVM pada voice.csv ===")
display(voice_results_df)

# Tabel perbandingan akurasi (pivot) antar kernel
accuracy_table = voice_results_df.pivot(index="Split", columns="Kernel", values="Accuracy (%)")
accuracy_table = accuracy_table[["linear", "poly", "rbf"]]
print("\n=== Tabel Akurasi (%) ===")
display(accuracy_table)

=== Ringkasan Accuracy SVM pada voice.csv ===


,Split,Kernel,Train,Test,Accuracy,Accuracy (%)
0,70:30,linear,2217,951,0.978970,97.90
1,70:30,poly,2217,951,0.960042,96.00
2,70:30,rbf,2217,951,0.983176,98.32
3,80:20,linear,2534,634,0.974763,97.48
4,80:20,poly,2534,634,0.955836,95.58
5,80:20,rbf,2534,634,0.982650,98.26



=== Tabel Akurasi (%) ===


Kernel,linear,poly,rbf
Split,,,
70:30,97.90,96.00,98.32
80:20,97.48,95.58,98.26


In [4]:
# Detail per model: confusion matrix & classification report model terbaik (rbf, 70:30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

best_voice_model = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
best_voice_model.fit(X_train, y_train)
y_pred = best_voice_model.predict(X_test)

print("=== Model terbaik voice.csv: SVC(kernel='rbf'), split 70:30 ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=["female", "male"]))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["female", "male"]))

=== Model terbaik voice.csv: SVC(kernel='rbf'), split 70:30 ===
Confusion Matrix:
[[467   9]
 [  7 468]]

Classification Report:
              precision    recall  f1-score   support

      female       0.99      0.98      0.98       476
        male       0.98      0.99      0.98       475

    accuracy                           0.98       951
   macro avg       0.98      0.98      0.98       951
weighted avg       0.98      0.98      0.98       951



## Bagian 2 — Klasifikasi Siang/Malam

In [5]:
# Kumpulkan seluruh gambar daytime & nighttime (training + test) dari Praktikum 5
image_files = sorted(
    p
    for p in image_root.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and not p.name.startswith("._")          # abaikan file resource-fork macOS
    and p.parent.name in {"day", "night"}
)

y_img = np.array([p.parent.name for p in image_files])

print("Jumlah gambar :", len(y_img))
print("Distribusi kelas:")
print(pd.Series(y_img).value_counts())

Jumlah gambar : 400
Distribusi kelas:
day      200
night    200
Name: count, dtype: int64


In [6]:
def extract_hsv_histogram(image_path, bins=32, size=(256, 256)):
    # Ekstraksi fitur histogram HSV (32 bin per kanal) dari sebuah gambar
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Gambar tidak dapat dibaca: {image_path}")

    image = cv2.resize(image, size)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    features = []
    for channel in range(3):
        value_range = [0, 180] if channel == 0 else [0, 256]
        hist = cv2.calcHist([hsv], [channel], None, [bins], value_range)
        hist = cv2.normalize(hist, hist).flatten()
        features.extend(hist)

    return features


X_hist = np.asarray([extract_hsv_histogram(path) for path in image_files])
y_img = np.asarray([p.parent.name for p in image_files])

print("Shape fitur histogram:", X_hist.shape)
print("Contoh nilai fitur (5 fitur pertama):", X_hist[0][:5])

Shape fitur histogram: (400, 96)
Contoh nilai fitur (5 fitur pertama): [0.00406189 0.00638297 0.00842653 0.03648134 0.0176604 ]


In [7]:
# Split 80:20 stratified untuk klasifikasi siang/malam
X_train, X_test, y_train, y_test = train_test_split(
    X_hist, y_img, test_size=0.20, random_state=42, stratify=y_img
)

print("Data latih :", X_train.shape, pd.Series(y_train).value_counts().to_dict())
print("Data uji   :", X_test.shape, pd.Series(y_test).value_counts().to_dict())

Data latih : (320, 96) {'night': 160, 'day': 160}
Data uji   : (80, 96) {'day': 40, 'night': 40}


In [8]:
# --- Hyperparameter tuning kernel RBF (GridSearchCV 5-fold pada data latih) ---
param_grid = {
    "svc__C": [0.1, 1, 10, 100, 1000],
    "svc__gamma": ["scale", 1e-4, 1e-3, 1e-2, 1e-1],
}

svc_pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf"))

grid_search = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)

y_pred = grid_search.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=== Tuning Kernel RBF (80:20) ===")
print("Best params :", grid_search.best_params_)
print(f"CV score    : {grid_search.best_score_ * 100:.2f}%")
print(f"Accuracy uji: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["day", "night"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=["day", "night"]))

=== Tuning Kernel RBF (80:20) ===
Best params : {'svc__C': 1, 'svc__gamma': 'scale'}
CV score    : 100.00%
Accuracy uji: 100.00%

Classification Report:
              precision    recall  f1-score   support

         day       1.00      1.00      1.00        40
       night       1.00      1.00      1.00        40

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

Confusion Matrix:
[[40  0]
 [ 0 40]]


In [9]:
# Perbandingan manual beberapa kombinasi (C, gamma) pada data uji
tuning_results = []
for C in [0.1, 1, 10, 100, 1000]:
    for gamma in ["scale", 1e-3, 1e-2, 1e-1]:
        model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma=gamma))
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        tuning_results.append({"C": C, "Gamma": gamma, "Accuracy": acc})

tuning_df = pd.DataFrame(tuning_results)
tuning_df["Accuracy (%)"] = (tuning_df["Accuracy"] * 100).round(2)

print("=== Accuracy tiap kombinasi (C, gamma) pada data uji 80:20 ===")
display(tuning_df.pivot(index="C", columns="Gamma", values="Accuracy (%)"))

=== Accuracy tiap kombinasi (C, gamma) pada data uji 80:20 ===


Gamma,0.001,0.01,0.1,scale
C,,,,
0.1,86.25,98.75,72.5,98.75
1.0,100.00,100.00,92.5,100.00
10.0,100.00,100.00,92.5,100.00
100.0,98.75,100.00,92.5,100.00
1000.0,98.75,100.00,92.5,100.00


In [10]:
# Uji GENERALISASI tambahan: latih pada folder 'training' dan uji pada folder 'test'
fold = np.array([p.parent.parent.name for p in image_files])

train_idx = np.where(fold == "training")[0]
test_idx = np.where(fold == "test")[0]

folder_model = GridSearchCV(
    estimator=make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
folder_model.fit(X_hist[train_idx], y_img[train_idx])
folder_acc = accuracy_score(y_img[test_idx], folder_model.predict(X_hist[test_idx]))

print("=== Generalisasi pada split folder asli Praktikum 5 (training -> test) ===")
print("Best params:", folder_model.best_params_)
print(f"Accuracy   : {folder_acc * 100:.2f}%")

=== Generalisasi pada split folder asli Praktikum 5 (training -> test) ===
Best params: {'svc__C': 10, 'svc__gamma': 0.001}
Accuracy   : 97.50%


## Ringkasan Hasil

In [11]:
# Ringkasan seluruh hasil
summary_rows = []

for _, row in voice_results_df.iterrows():
    summary_rows.append(
        {
            "Tugas": "1 - voice.csv",
            "Data": "Gender (male/female)",
            "Split": row["Split"],
            "Kernel": row["Kernel"],
            "Accuracy (%)": row["Accuracy (%)"],
        }
    )

summary_rows.append(
    {
        "Tugas": "2 - gambar Praktikum 5",
        "Data": "Day/Night (histogram HSV)",
        "Split": "80:20",
        "Kernel": f"rbf (C={grid_search.best_params_['svc__C']}, gamma={grid_search.best_params_['svc__gamma']})",
        "Accuracy (%)": round(accuracy * 100, 2),
    }
)

summary_rows.append(
    {
        "Tugas": "2 - gambar Praktikum 5",
        "Data": "Day/Night (histogram HSV)",
        "Split": "training -> test",
        "Kernel": f"rbf (C={folder_model.best_params_['svc__C']}, gamma={folder_model.best_params_['svc__gamma']})",
        "Accuracy (%)": round(folder_acc * 100, 2),
    }
)

summary_df = pd.DataFrame(summary_rows)
summary_df

,Tugas,Data,Split,Kernel,Accuracy (%)
0,1 - voice.csv,Gender (male/female),70:30,linear,97.90
1,1 - voice.csv,Gender (male/female),70:30,poly,96.00
2,1 - voice.csv,Gender (male/female),70:30,rbf,98.32
3,1 - voice.csv,Gender (male/female),80:20,linear,97.48
4,1 - voice.csv,Gender (male/female),80:20,poly,95.58
5,1 - voice.csv,Gender (male/female),80:20,rbf,98.26
6,2 - gambar Praktikum 5,Day/Night (histogram HSV),80:20,"rbf (C=1, gamma=scale)",100.00
7,2 - gambar Praktikum 5,Day/Night (histogram HSV),training -> test,"rbf (C=10, gamma=0.001)",97.50
